# Fine Tuning SAM Audio

```mermaid
flowchart TD

    subgraph Data Preparation Phase
        RawStems[Raw Instrument Stems: Bağlama and Zurna] --> DynamicMix[Dynamic Mixing Strategy]
        DynamicMix --> MixtureAudio[Complex Mixture Audio]
        DynamicMix --> TargetAudio[Isolated Ground Truth Audio]
        Prompt[Text Prompt e.g., 'bağlama solo'] --> Tuple[Create Training Tuple]
        MixtureAudio --> Tuple
        TargetAudio --> Tuple
    end

    subgraph Latent Compression Phase DAC-VAE
        Tuple --> LoadVAE[Extract SAM Audio Built-in VAE]
        LoadVAE --> FreezeVAE[Freeze VAE Weights: No Gradients]
        FreezeVAE --> EncodeLatents[Encode Audio to Continuous Latents]
        EncodeLatents --> LatentTensors[Output 128-channel .pt Tensors]
    end

    subgraph Custom PyTorch Setup Phase
        LatentTensors --> CustomDataset[Build Custom PyTorch Dataset Class]
        CustomDataset --> LoadBase[Load Frozen Base: facebook/sam-audio-large]
        LoadBase --> InjectLoRA[Wrap with PEFT/LoRA: wq, wv]
        InjectLoRA --> EnableGrads[Enable Checkpointing & Input Requires Grad]
        EnableGrads --> LoadScheduler[Init diffusers FlowMatchEulerDiscreteScheduler]
        LoadScheduler --> Accelerate[Setup accelerate for VRAM Management]
    end

    subgraph Custom LoRA Flow-Matching Training Loop
        Accelerate --> SqueezeBatch[Fix Double Batch: Apply .squeeze 1 & .transpose]
        SqueezeBatch --> MetaTrick[Meta's Trick: Duplicate 128 channels to 256 via torch.cat]
        MetaTrick --> TimeSample[Sample Random Continuous Timestep: t]
        TimeSample --> AddNoise[Noise 256-channel Target Latent to create state: x_t_256]
        AddNoise --> ManualWedge[Manual Tensor Wedge: x_t_256.requires_grad_ True]
        ManualWedge --> ForwardPass[Forward Pass: DiT conditioned on x_t, t, Mixture, Text]
        ForwardPass --> CalcLoss[Calculate Vector Field MSE Loss in 256-channel Space]
        CalcLoss --> Backprop[Backpropagate Gradients strictly to LoRA Adapters]
        Backprop --> SaveAdapter[Export adapter_model.safetensors]
    end

    subgraph Inference and Validation Phase
        SaveAdapter --> DeployModel[Load Base Model + Injected LoRA Adapter]
        DeployModel --> InputNovel[Input Novel Unseen Audio Mixture & Prompt]
        InputNovel --> DiTSeparation[DiT Generates Continuous 256-channel Target Latents]
        DiTSeparation --> SliceLatents[Slice Latents Back to 128 Channels]
        SliceLatents --> DecodeAudio[DAC-VAE Decoder Reconstructs Isolated Audio]
    end
```

## Load SAM Audio and Processor

In [1]:
import torch
from sam_audio import SAMAudio, SAMAudioProcessor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_ID = "facebook/sam-audio-base"  # base model

processor = SAMAudioProcessor.from_pretrained(MODEL_ID)
base_model = SAMAudio.from_pretrained(MODEL_ID)
base_model = base_model.to(device=device, dtype=torch.float16)

base_model.eval()  # Freeze model

/teamspace/studios/this_studio/turkish-instrument/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/teamspace/studios/this_studio/turkish-instrument/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.11.0+cu130)
    Python  3.10.19 (you have 3.11.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be

SAMAudio(
  (audio_codec): DACVAE(
    (encoder): Encoder(
      (block): Sequential(
        (0): NormConv1d(1, 64, kernel_size=(7,), stride=(1,), padding=(3,))
        (1): EncoderBlock(
          (block): Sequential(
            (0): ResidualUnit(
              (block): Sequential(
                (0): Snake1d()
                (1): NormConv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(3,))
                (2): Snake1d()
                (3): NormConv1d(64, 64, kernel_size=(1,), stride=(1,))
              )
            )
            (1): ResidualUnit(
              (block): Sequential(
                (0): Snake1d()
                (1): NormConv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(9,), dilation=(3,))
                (2): Snake1d()
                (3): NormConv1d(64, 64, kernel_size=(1,), stride=(1,))
              )
            )
            (2): ResidualUnit(
              (block): Sequential(
                (0): Snake1d()
                (1): NormConv1d(64, 64, 

## Data Preparation

In [ ]:
import os
import pandas as pd
import torch
import torchaudio

# Load metadata
metadata_path = '../data/mixed/metadata.csv'
df = pd.read_csv(metadata_path)

# Initialize paths for latents
dirs = {
    'mixed': '../data/latents/mixed/',
    'baglama': '../data/latents/baglama/',
    'zurna': '../data/latents/zurna/'
}
for d in dirs.values():
    os.makedirs(d, exist_ok=True)

## Latent Compression

In [ ]:
# Load frozen builtin VAE model
vae = base_model.audio_codec

vae.eval()  # Freeze VAE
for param in vae.parameters():
    param.requires_grad = False


def process_audio(file_path, model_sample_rate):
    try:
        wav, sample_rate = torchaudio.load(file_path)

        # Ensure [C, T]
        if wav.ndim == 1:
            wav = wav.unsqueeze(0)  # [1, T]
        elif wav.ndim == 2 and wav.shape[0] > wav.shape[1]:
            wav = wav.transpose(0, 1)  # [T, C] -> [C, T]

        wav = wav.float()

        orig_sr = int(sample_rate)
        target_sr = int(processor.audio_sampling_rate)

        # Resample
        if sample_rate != model_sample_rate:
            wav = torchaudio.functional.resample(wav, orig_sr, target_sr)

        # Mono
        wav = wav.mean(0, keepdim=True)  # [1, T]

        # Expected shape is batch x 1 x samples
        wav = wav.unsqueeze(0).to(device=device, dtype=next(base_model.parameters()).dtype)

        return wav
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None


In [ ]:
from tqdm.auto import tqdm

print("Encoding audio to latents...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    mixed_file = row['file_name']
    zurna_source = row['zurna_source']
    baglama_source = row['baglama_source']

    # Resolve paths
    mixed_path = os.path.join('../data/mixed/', mixed_file)
    zurna_path = os.path.join('../data/processed/zurna/', zurna_source)
    baglama_path = os.path.join('../data/processed/baglama/', baglama_source)

    # Process and encode each category
    file_mappings = [
        ('mixed', mixed_path, mixed_file),
        ('zurna', zurna_path, zurna_source),
        ('baglama', baglama_path, baglama_source)
    ]

    for label, audio_path, filename in file_mappings:
        base_name = filename.replace('.wav', '.pt')
        out_path = os.path.join(dirs[label], base_name)

        # Skip if already encoded
        if os.path.exists(out_path):
            continue

        wav_input = process_audio(audio_path, processor.audio_sampling_rate)
        if wav_input is not None:
            with torch.no_grad():
                encoded = vae(wav_input)
            torch.save(encoded.cpu(), out_path)

print("Latent encoding complete!")

Encoding audio to latents...


100%|██████████| 60/60 [00:12<00:00,  4.63it/s]

Latent encoding complete!


## Setup PEFT/LoRA Fine-Tune

In [13]:
base_model.text_encoder.requires_grad_(False)

with torch.no_grad():
    text_features, text_mask = base_model.text_encoder(['zurna'])

print("Text embeddings shape:", text_features.shape)

Text embeddings shape: torch.Size([1, 4, 768])


### Custom Dataset

In [30]:
from torch.utils.data import Dataset
import torch
import pandas as pd
import os

class TurkishMusicDataset(Dataset):
    def __init__(self, metadata_csv, latent_dirs, text_encoder):
        self.df = pd.read_csv(metadata_csv)
        self.latent_dirs = latent_dirs
        self.instruments = ['zurna', 'baglama']

        # Precompute embeddings once per instrument
        self.text_embeddings = {}
        with torch.no_grad():
            for inst in self.instruments:
                features, mask = text_encoder([inst])
                # Squeeze out the batch dimension to store single items
                self.text_embeddings[inst] = {
                    'features': features.squeeze(0),
                    'mask': mask.squeeze(0)
                }

    def __len__(self):
        # We double the length because each row yields two samples (zurna and baglama)
        return len(self.df) * 2

    def __getitem__(self, idx):
        # Determine which row and parameter to use based on index
        row_idx = idx // 2
        instrument = self.instruments[idx % 2]

        row = self.df.iloc[row_idx]

        # Load mix latent
        mixed_filename = row['file_name']
        mixed_base = mixed_filename.replace('.wav', '.pt')
        mix_latent = torch.load(os.path.join(self.latent_dirs, 'mixed', mixed_base)).squeeze(0)

        # Load target latent
        target_filename = row[f'{instrument}_source']
        target_base = target_filename.replace('.wav', '.pt')
        target_latent = torch.load(os.path.join(self.latent_dirs, instrument, target_base)).squeeze(0)

        return {
            'mix_latent': mix_latent,
            'target_latent': target_latent,
            'text_features': self.text_embeddings[instrument]['features'],
            'text_mask': self.text_embeddings[instrument]['mask'],
            'instrument': instrument
        }

In [31]:
tmd = TurkishMusicDataset('../data/mixed/metadata.csv', '../data/latents/', base_model.text_encoder)
print(f"Dataset size: {len(tmd)}")
sample = tmd[0]
print("Sample keys:", sample.keys())
print(f"Instrument: {sample['instrument']}")

Dataset size: 120
Sample keys: dict_keys(['mix_latent', 'target_latent', 'text_features', 'text_mask', 'instrument'])
Instrument: zurna


In [32]:
sample['mix_latent'].shape, sample['target_latent'].shape, sample['text_features'].shape, sample['text_mask'].shape

(torch.Size([128, 125]),
 torch.Size([128, 125]),
 torch.Size([4, 768]),
 torch.Size([4]))

In [33]:
from torch.utils.data import random_split, DataLoader
import torch

# 1. Define the size of the splits (e.g., 80% for training, 20% for testing)
train_size = int(0.8 * len(tmd))
test_size = len(tmd) - train_size

# 2. Split the dataset (using a generator for reproducibility)
train_dataset, test_dataset = random_split(
    tmd,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)  # Optional: ensures the same split every run
)

print(f"Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")

# 3. Create the DataLoaders
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

Train size: 96 | Test size: 24


### Wrap with PEFT/LoRA

In [52]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    # The developers of SAMAudio named the attention layers in their DiT (the DiT handles the actual latent generation) wq and wv
    target_modules=["wq", "wv"],
    lora_dropout=0.05,
    bias="none"
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

/teamspace/studios/this_studio/turkish-instrument/.venv/lib/python3.11/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/teamspace/studios/this_studio/turkish-instrument/.venv/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 10,690,560 || all params: 6,474,855,786 || trainable%: 0.1651


### Initialize Flow Matching Scheduler and Accelerate

In [53]:
NUM_EPOCHS = 100

In [54]:
from diffusers import FlowMatchEulerDiscreteScheduler
from diffusers.optimization import get_cosine_schedule_with_warmup
from accelerate import Accelerator
from torch.optim import AdamW
import math

# initialize flow-matching scheduler
scheduler = FlowMatchEulerDiscreteScheduler(
    num_train_timesteps=1000,
    shift=1.0 # standard shift value for flow matching
)

# initialize accelerator for vram and device management
accelerator = Accelerator(
    gradient_accumulation_steps=16,  # accumulate gradients over 4 steps
    mixed_precision="fp16"
)

optimizer = AdamW(peft_model.parameters(), lr=1e-4)

# CRITICAL: Because we are accumulating gradients over 16 batches,
# the optimizer only takes a "step" once every 16 forward passes.
num_update_steps_per_epoch = math.ceil(len(train_loader) / accelerator.gradient_accumulation_steps)
max_train_steps = NUM_EPOCHS * num_update_steps_per_epoch

# 2. Define Warmup Steps
# A standard rule of thumb for LoRA is warming up for 5% to 10% of total training steps.
num_warmup_steps = int(max_train_steps * 0.10)

# 3. Initialize the Scheduler
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=max_train_steps
)

# Pass everything to accelerator.prepare
peft_model, optimizer, train_loader, test_loader, lr_scheduler = accelerator.prepare(
    peft_model, optimizer, train_loader, test_loader, lr_scheduler
)

print("Flow-Matching Scheduler and Accelerator Initialized!")
print(f"Total Training Steps: {max_train_steps}")
print(f"Warmup Steps: {num_warmup_steps}")

Flow-Matching Scheduler and Accelerator Initialized!
Total Training Steps: 600
Warmup Steps: 60


## Custom LoRA Flow-Matching Training Loop

In [55]:
import inspect

# Inspect the forward method of the underlying base model
signature = inspect.signature(base_model.forward)

print("SAM Audio Forward Signature:")
for param in signature.parameters.values():
    print(f"- {param.name}: {param.default}")

SAM Audio Forward Signature:
- noisy_audio: <class 'inspect._empty'>
- audio_features: <class 'inspect._empty'>
- text_features: <class 'inspect._empty'>
- time: <class 'inspect._empty'>
- masked_video_features: None
- text_mask: None
- anchor_ids: None
- anchor_alignment: None
- audio_pad_mask: None


In [56]:
import torch
import torch.nn.functional as F

# Ensure the frozen base model is in eval mode, but PEFT adapters are training
peft_model.train()

for epoch in range(NUM_EPOCHS):
    # train_dataloader yields the Tuple: Mixture, Target, Text Embeddings
    for step, batch in enumerate(train_loader):

        # accelerate.accumulate handles the 16-step gradient hold for your T4
        with accelerator.accumulate(peft_model):

            # 1. Unpack the Pre-Computed Tensors
            mixture_latent = batch["mix_latent"].transpose(1, 2)         # Conditioner 1: [Batch, 125, 128]
            target_latent = batch["target_latent"].transpose(1, 2)           # Ground Truth x_1: [Batch, 125, 128]
            text_features = batch["text_features"]    # Conditioner 2: Frozen Embeddings
            text_mask = batch["text_mask"]

            batch_size = target_latent.shape[0]

            # 2. Generate Pure Gaussian Noise (x_0)
            # noise = torch.randn_like(target_latent)

            # 3. Time Sampling
            # You randomly select a continuous timestep (t) between 0 and 1[cite: 1062].
            t = torch.rand((batch_size,), device=accelerator.device)
            t_expanded = t.view(batch_size, 1, 1) # Reshape for tensor broadcasting

            # 4. Noising the Destination (The Probability Path)
            # You use the scheduler to dynamically add noise to your Target Latent, creating a degraded state (x_t)[cite: 1063].
            # In standard Flow-Matching, this is a straight mathematical line[cite: 1096].
            # x_t = (1 - t_expanded) * noise + t_expanded * target_latent

            # additional step, Meta's trick (https://github.com/facebookresearch/sam-audio/blob/68b48d48fff1ad776d3afefbe634eb5f5d60ba7b/sam_audio/model/model.py#L184)
            mixture_256 = torch.cat([mixture_latent, mixture_latent], dim=-1)
            target_256 = torch.cat([target_latent, target_latent], dim=-1)
            # generate Noise and Noisy State in the 256-dimensional space
            noise_256 = torch.randn_like(target_256)
            x_t_256 = (1 - t_expanded) * noise_256 + t_expanded * target_256

            # 5. The Forward Pass
            # Wrap the forward pass to automatically handle mixed precision casting
            with accelerator.autocast():
                # You feed the noisy state (x_t), the timestep (t), and your two completely clean conditioners (the Mixture Latent and the Text Prompt) into your peft_model[cite: 1064].
                predicted_velocity = peft_model(
                    noisy_audio=x_t_256,
                    time=t,
                    audio_features=mixture_256,
                    text_features=text_features,
                    text_mask=text_mask
                )

                # 6. Calculate the True Vector Field
                # Flow-matching models predict the trajectory pointing from noise to target.
                true_velocity = target_256 - noise_256

                # 7. Vector Field Discrepancy Loss
                # Calculate the MSE between the predicted trajectory and the true trajectory[cite: 1107].
                loss = F.mse_loss(predicted_velocity, true_velocity)

            # 8. The Update
            # Push the gradients back strictly to the tiny q_proj and v_proj LoRA weights[cite: 1066].
            accelerator.backward(loss)

            # Optimizer automatically respects the gradient_accumulation_steps=16 we set earlier
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        # Optional: Print loss to verify convergence
        if step % 50 == 0:
            print(f"Epoch {epoch} | Step {step} | Flow-Matching Loss: {loss.item()}")

Epoch 0 | Step 0 | Flow-Matching Loss: 1.2921477556228638


Epoch 0 | Step 50 | Flow-Matching Loss: 0.6412034034729004
Epoch 1 | Step 0 | Flow-Matching Loss: 0.59209805727005
Epoch 1 | Step 50 | Flow-Matching Loss: 1.1883983612060547
Epoch 2 | Step 0 | Flow-Matching Loss: 1.3387641906738281
Epoch 2 | Step 50 | Flow-Matching Loss: 1.334055781364441
Epoch 3 | Step 0 | Flow-Matching Loss: 0.498209685087204
Epoch 3 | Step 50 | Flow-Matching Loss: 1.2786047458648682
Epoch 4 | Step 0 | Flow-Matching Loss: 1.193366527557373
Epoch 4 | Step 50 | Flow-Matching Loss: 0.5136173367500305
Epoch 5 | Step 0 | Flow-Matching Loss: 1.1465271711349487
Epoch 5 | Step 50 | Flow-Matching Loss: 0.6730529069900513
Epoch 6 | Step 0 | Flow-Matching Loss: 0.8680023550987244
Epoch 6 | Step 50 | Flow-Matching Loss: 0.3834601044654846
Epoch 7 | Step 0 | Flow-Matching Loss: 0.6836830377578735
Epoch 7 | Step 50 | Flow-Matching Loss: 0.6581822633743286
Epoch 8 | Step 0 | Flow-Matching Loss: 0.8148791193962097
Epoch 8 | Step 50 | Flow-Matching Loss: 0.578116238117218
Epoch 9 | S

KeyboardInterrupt: 